In [10]:
import re, os, PyPDF2
import unicodedata

def anonimizar_documento(texto):

    # ==============================
    # NORMALIZAÇÃO
    # ==============================
    def normalizar(texto):
        texto = texto.replace('\xa0', ' ')
        texto = unicodedata.normalize('NFKD', texto)
        texto = texto.encode('ASCII', 'ignore').decode('ASCII')
        return texto

    # ==============================
    # FUNÇÕES DE ANONIMIZAÇÃO
    # ==============================

    def anonymize_remover_cabecalhos(texto):
        padroes = [
            r"Assinado digitalmente por.*",
            r"Documento assinado eletronicamente.*",
            r"Protocolo:\s*\d+",
            r"URL para download:.*",
            r"Hash de autenticação:.*",
        ]
        for padrao in padroes:
            texto = re.sub(padrao, "", texto, flags=re.IGNORECASE)
        return texto

    def anonymize_cpfs(texto):
        return re.sub(
            r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b",
            "[CPF]",
            texto
        )

    def anonymize_cnpj(texto):
        return re.sub(
            r"\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b",
            "[CNPJ]",
            texto
        )

    def anonymize_processos(texto):
        return re.sub(
            r"\b\d{9}\b",
            "[PROCESSO]",
            texto
        )

    def anonymize_emails(texto):
        return re.sub(
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
            "[EMAIL]",
            texto
        )

    def anonymize_documentos_identificacao(texto):
        regex = r"(CPF\/CNPJ|CPF|CNPJ)\s*:\s*([A-Z0-9\-\.\/]+)"
        return re.sub(
            regex,
            "[DOC_ID]",
            texto,
            flags=re.IGNORECASE
        )

    def anonymize_endereco(texto):
        regex = r"(Endere[cç]o\s*:\s*)(.+)"
        return re.sub(
            regex,
            "[ENDERECO]",
            texto,
            flags=re.IGNORECASE
        )

    # ==============================
    # REMOVER A LINHA INTEIRA QUE TENHA CEP
    # ==============================
    def anonymize_remover_linhas_com_cep(texto):
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'\bCEP\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)
    
    # ==============================
    # REMOVER CABEÇALHO DAS PÁGINAS DA PETIÇÃO
    # ==============================
    def anonymize_remover_cabecalhos_pagina(texto):
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'^\s*Peticao\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)
        
    def anonymize_telefone(texto):
        regex = r"\b(Fone\/Fax|Fone|Telefone|Tel\.?|Fax)\b\s*:?\s*([\(\d][\d\.\-\)\s]+)"
        return re.sub(
            regex,
            "[TELEFONE]",
            texto,
            flags=re.IGNORECASE
        )

    def anonymize_nomes_rotulados(texto):
        regex = r"(Requerente|Tecnico|Inventor|Procurador)\s*:\s*([A-ZÁÉÍÓÚÂÊÔÃÕÇ\s]+)"
        return re.sub(
            regex,
            "[PESSOA_NATURAL]",
            texto
        )

    # ==============================
    # ELIMINA PÁGINAS INICIAIS
    # ==============================

    def iniciar_apos_recurso(texto):
        padrao = r"(RECURSO\s+do\s+despacho\s+que\s+indeferiu\s+o\s+Pedido\s+de\s+Paten\s*te|Excelentissimo|Ilmo\s+Senhor\s+presidente|recurso\s+ao\s+presidente\s+|apresentar\s+recurso\s+desta\s+decis[aã]o|Em\s+resposta\s+a\s*(?:o|ao)\s+Indeferi\s*-?\s*mento)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO\s*$"
            match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
            if match:
                return texto[match.start():]
            else:
                padrao = r"^\s*RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                if match:
                    return texto[match.start():]
                else:
                    padrao = r"^\s*INTERPOSICAO\s+DE\s+RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                    if match:
                        return texto[match.start():]
                    else:
                        padrao = r"(recurso\s+contra\s+o\s+indeferimento|recurso\s+ao\s+indeferimento|recurso\s+contra\s+decisao\s+de\s+indeferimento)"
                        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                        if match:
                            return texto[match.start():]
                        else:
                            padrao = r"(ilustrissimos\s+examinadores)"
                            match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                            if match:
                                return texto[match.start():]
                            else:
                                padrao = r"(recurso\s+que\s+bastante\s+faz)"
                                match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                                if match:
                                    return texto[match.start():]
                                return ""
 
    # ==============================
    # PIPELINE
    # ==============================

    texto = normalizar(texto)
    texto = iniciar_apos_recurso(texto)
    texto = anonymize_remover_cabecalhos(texto)
    texto = anonymize_cpfs(texto)
    texto = anonymize_cnpj(texto)
    texto = anonymize_emails(texto)
    texto = anonymize_processos(texto)
    texto = anonymize_documentos_identificacao(texto)
    texto = anonymize_endereco(texto)
    texto = anonymize_telefone(texto)
    texto = anonymize_nomes_rotulados(texto)
    texto = anonymize_remover_linhas_com_cep(texto)
    texto = anonymize_remover_cabecalhos_pagina(texto)

    return texto

arquivo = 'MU9100581_29409162312309455_214.pdf' # Recurso contra o indeferimento
arquivo = 'MU9100724_29409162317696511_214.pdf' # Recurso ao indeferimento, gera varios espaços vazios porque tem imagens embutidas no texto


nome_sem_extensao = arquivo.replace('.pdf', '')
partes = nome_sem_extensao.split('_')
numero = partes[0]
numnossonumero = partes[1]

file_path = f"pareceres/peticoes/{numero}_{numnossonumero}_214.pdf"

documento = ''
if os.path.exists(file_path):
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            documento += page.extract_text() or ""
else:
    print(f"Arquivo não encontrado: {file_path}")
    
texto_anon = anonimizar_documento(documento)
print(texto_anon)


 
 
   
RECURSO AO INDEFERIMENTO  
 
 
 
Entende o Recorrente  estar o Pedido de Patente em tela de acordo com as 
recomendacoes da Lei da Propriedade Industrial n.o 9.279, de 14/05/1996 , conforme fatos e 
fundamentos apresentados a seguir:  
 
I  DO PARECER TECNICO  DE 
INDEFERIMENTO  
 
 
1. O referido parecer tecnico apesenta resumida mente as seguintes consideracoes , 
concluindo pelo nao atendimento aos art. 9o e 1 4 da LPI : 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2. Logo, conclui o Sr. Examinador que o objeto em analise, no que tange a sua 
forma construtiva,  esta na ordem natural das coisas a um tecnic o no assunto  pelos documentos 
citados, o que destitui o pedido de ato inventiv o frente a D1.  
 
3. Tem -se ainda que o novo quadro reivindicatorio foi aceito por nao contrariar a 
normativa vigente, nao possuindo acrescimo de materia, mas possui trechos que nao se 
configuram como uma nova forma ou disposicao.  
 
 
 
 
 
 
 
 
 
II  DO OBJETO DA PATENTE FRENTE AS AN